# Load trained controllers

Use this notebook to rebuild any saved controller from its YAML config, load the matching `.ckpt`, switch it to `eval()`, and inspect the layers/weights before exporting to C.

In [1]:
from pathlib import Path
import sys

import torch
import numpy as np

# Works whether the notebook is opened from the repo root or from this folder.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "LNN_behavioural_cloning_quadrotor" else NOTEBOOK_DIR / "LNN_behavioural_cloning_quadrotor"

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Could not find project folder from {NOTEBOOK_DIR}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.config import load_yaml, resolve_checkpoint, resolve_saved_config, dataset_dims
from utils.model_builder import build_controller_network
from utils.lightning import Lightning_Model

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CONFIG_DIR = PROJECT_ROOT / "configs"

print("Project root:", PROJECT_ROOT)
print("Torch:", torch.__version__)

Project root: /home/gustavokpc/Documents/ESTAG/LNN_estag/LNN_behavioural_cloning_quadrotor
Torch: 2.12.0+cu130


In [2]:
def infer_legacy_config(config, model_name):
    """
    Some older YAMLs in configs/ do not store model.type/backbone_units.
    Infer those fields from the checkpoint name so the saved state_dict fits.
    """
    config = dict(config)
    config["model"] = dict(config.get("model", {}))

    name = model_name.lower()
    model_cfg = config["model"]

    if "type" not in model_cfg:
        if "gru" in name:
            model_cfg["type"] = "gru"
        elif "lstm" in name:
            model_cfg["type"] = "lstm"
        elif "ctrnn" in name:
            model_cfg["type"] = "ctrnn"
        elif "rnn" in name:
            model_cfg["type"] = "simplernn"
        elif "ncp" in name:
            model_cfg["type"] = "ncp"
        elif "cfc" in name:
            model_cfg["type"] = "cfc"
        elif "ltc" in name:
            model_cfg["type"] = "ltc"
        elif "mlp" in name:
            model_cfg["type"] = "mlp"

    if "cfc_pure" in name or "_pure_" in name:
        model_cfg["cfc_mode"] = "pure"
        model_cfg.setdefault("backbone_units", 128)

    return config


def load_controller(model_name, device="cpu", legacy_infer=True):
    """Load one trained controller from checkpoints/<name>.ckpt + configs/<name>.yaml."""
    device = torch.device(device)

    config_path = resolve_saved_config(model_name, CONFIG_DIR)
    ckpt_path = resolve_checkpoint(model_name, PROJECT_ROOT)
    config = load_yaml(config_path)
    if legacy_infer:
        config = infer_legacy_config(config, model_name)

    input_dim, output_dim = dataset_dims(config)
    network = build_controller_network(config, input_dim, output_dim)
    model = Lightning_Model(network, config)

    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint["state_dict"])
    model.to(device)
    model.eval()

    return model, network, config, ckpt_path, config_path


def print_model_summary(model, network, config, ckpt_path, config_path):
    print("Checkpoint:", ckpt_path.name)
    print("Config:", config_path.name)
    print("Model type:", config["model"]["type"])
    print("Input labels:", config["dataset"]["input_labels"])
    print("Output labels:", config["dataset"]["output_labels"])
    print("Input dim:", config["dataset"].get("input_dim", config["dataset"].get("input_size")))
    print("Output dim:", config["dataset"].get("output_dim", config["dataset"].get("output_size")))
    print("Sequencing:", config.get("sequencing", {}))
    print("\nNetwork:")
    print(network)


def print_leaf_layers(network):
    """Print modules with no children: Linear, ReLU, GRU, LSTM, CfC cells, etc."""
    for name, module in network.named_modules():
        if name and len(list(module.children())) == 0:
            print(f"{name}: {module}")


def print_weight_shapes(model_or_network):
    """Print parameter/buffer names and shapes. Use network for the cleanest names."""
    for name, tensor in model_or_network.state_dict().items():
        print(f"{name}: {tuple(tensor.shape)}")


def print_checkpoint_for_c_export(ckpt_name):
    """Load one checkpoint and print the same inspection blocks used for the MLP export."""
    model, network, config, ckpt_path, config_path = load_controller(ckpt_name)

    print("=" * 100)
    print_model_summary(model, network, config, ckpt_path, config_path)
    print("\nLeaf layers:")
    print_leaf_layers(network)
    print("\nWeight shapes:")
    print_weight_shapes(network)
    print()

    return model, network, config, ckpt_path, config_path

In [3]:
available_checkpoints = sorted(p.name for p in CHECKPOINT_DIR.glob("*.ckpt"))
available_checkpoints

['LTC_64_neurons_seq_1_epoch=18_val_loss=0.000193.ckpt',
 'RNN_64_neurons_seq_1_epoch=17_val_loss=0.000147.ckpt',
 'conv_cfc_default_n64_epoch=17_val_loss=0.000326.ckpt',
 'mlp_epoch=19_val_loss=0.003130.ckpt',
 'new_CFC_64_neurons_seq_1_epoch=18_val_loss=0.000142.ckpt',
 'new_CFC_pure_64_neurons_seq_1_epoch=17_val_loss=0.000203.ckpt',
 'new_CTRNN_64_neurons_seq_1_epoch=19_val_loss=0.000150.ckpt',
 'new_GRU_64_neurons_seq_1_epoch=19_val_loss=0.000088.ckpt',
 'new_LSTM_64_neurons_seq_1_epoch=17_val_loss=0.000092.ckpt',
 'new_NCP_CFC_60_neurons_seq_1_epoch=18_val_loss=0.000143.ckpt']

# To print a specific one (organized/individul way)

In [4]:
# SELECT MODEL_NAME HERE
MODEL_NAME = "mlp_epoch=19_val_loss=0.003130.ckpt"

model, network, config, ckpt_path, config_path = load_controller(MODEL_NAME)
print_model_summary(model, network, config, ckpt_path, config_path)

Checkpoint: mlp_epoch=19_val_loss=0.003130.ckpt
Config: mlp_epoch=19_val_loss=0.003130.yaml
Model type: mlp
Input labels: ['dx', 'dy', 'dz', 'vx', 'vy', 'vz', 'phi', 'theta', 'psi', 'p', 'q', 'r', 'Mx_ext', 'My_ext', 'Mz_ext', 'omega']
Output labels: ['u']
Input dim: 19
Output dim: 4
Sequencing: {'value': True, 'seq_len': 1}

Network:
FeedForwardSequenceController(
  (network): Sequential(
    (0): Linear(in_features=19, out_features=120, bias=True)
    (1): ReLU()
    (2): Linear(in_features=120, out_features=120, bias=True)
    (3): ReLU()
    (4): Linear(in_features=120, out_features=120, bias=True)
    (5): ReLU()
    (6): Linear(in_features=120, out_features=4, bias=True)
  )
)


In [5]:
# Layer list useful before writing C code.
print_leaf_layers(network)

network.0: Linear(in_features=19, out_features=120, bias=True)
network.1: ReLU()
network.2: Linear(in_features=120, out_features=120, bias=True)
network.3: ReLU()
network.4: Linear(in_features=120, out_features=120, bias=True)
network.5: ReLU()
network.6: Linear(in_features=120, out_features=4, bias=True)


In [6]:
# Shapes of the learned arrays. For C export, these are the tensors you will serialize.
print_weight_shapes(network)

network.0.weight: (120, 19)
network.0.bias: (120,)
network.2.weight: (120, 120)
network.2.bias: (120,)
network.4.weight: (120, 120)
network.4.bias: (120,)
network.6.weight: (4, 120)
network.6.bias: (4,)


# Print all "View as a scrollable element" to visualize all models 

In [8]:
# Print every checkpoint using the same inspection flow as the MLP cells above.
loaded_models = {}

for ckpt_name in available_checkpoints:
    try:
        m, n, cfg, ckpt, yaml_path = print_checkpoint_for_c_export(ckpt_name)
        loaded_models[ckpt_name] = {"model": m, "network": n, "config": cfg, "ckpt": ckpt, "yaml": yaml_path}
    except Exception as exc:
        print(f"ERROR {ckpt_name} -> {type(exc).__name__}: {exc}")

Checkpoint: LTC_64_neurons_seq_1_epoch=18_val_loss=0.000193.ckpt
Config: LTC_64_neurons_seq_1_epoch=18_val_loss=0.000193.yaml
Model type: ltc
Input labels: ['dx', 'dy', 'dz', 'vx', 'vy', 'vz', 'phi', 'theta', 'psi', 'p', 'q', 'r', 'Mx_ext', 'My_ext', 'Mz_ext', 'omega']
Output labels: ['u']
Input dim: 19
Output dim: 4
Sequencing: {'value': True, 'seq_len': 1}

Network:
ConvCfC(
  (conv_block): ConvBlock(
    (conv1): Conv1d(1, 64, kernel_size=(5,), stride=(2,), padding=(2,))
    (conv2): Conv1d(64, 128, kernel_size=(5,), stride=(2,), padding=(2,))
    (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (conv3): Conv1d(128, 128, kernel_size=(5,), stride=(2,), padding=(2,))
    (conv4): Conv1d(128, 256, kernel_size=(5,), stride=(2,), padding=(2,))
    (bn4): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
  (rnn): LTC(
    (rnn_cell): LTCCell(
      (make_positive_fn): Softplus(beta=1.0, thr